# **Final Demo Notebook**

## **Data Preparation**

### **Load the dataset**

In [47]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../Datasets/dataset.csv')

### **Clean the dataset (Drop uneeded columns and extreme outliers)**

In [ ]:
cols_to_drop = ["Unnamed: 0", "track_id", "mode"]
cleaned_df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
cleaned_df = cleaned_df.dropna()

numeric_cols = cleaned_df.select_dtypes(include=[np.number]).columns

Q1 = cleaned_df[numeric_cols].quantile(0.25)
Q3 = cleaned_df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

df_clean = cleaned_df[~((cleaned_df[numeric_cols] < (Q1 - 1.5 * IQR)) |
                (cleaned_df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]
df_clean.reset_index(drop=True, inplace=True)

if 'track_name' in df_clean.columns and 'artists' in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=['track_name', 'artists']).reset_index(drop=True)

### **Perform train test split**

In [ ]:
# Train/test split
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

full_feature_cols = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", "duration_ms"
 ]

selected_feature_cols = [
    "danceability", "energy", "loudness", "acousticness",
    "instrumentalness", "valence", "tempo", "duration_ms"
 ]


def prepare_scaled_matrices(df_train, df_eval, feature_cols):
    df_train_feat = df_train.dropna(subset=feature_cols).reset_index(drop=True)
    df_eval_feat = df_eval.dropna(subset=feature_cols).reset_index(drop=True)

    X_train_num = df_train_feat[feature_cols].values
    X_eval_num = df_eval_feat[feature_cols].values

    genre_cols = [c for c in df_train_feat.columns if 'genre' in c.lower()]
    if genre_cols:
        genre_col = genre_cols[0]
        train_genre = df_train_feat[genre_col].astype('category')
        eval_genre = df_eval_feat[genre_col].astype('category')
        train_dummies = pd.get_dummies(train_genre, prefix=genre_col)
        eval_dummies = pd.get_dummies(eval_genre, prefix=genre_col)
        eval_dummies = eval_dummies.reindex(columns=train_dummies.columns, fill_value=0)
        X_train = np.hstack([X_train_num, train_dummies.values])
        X_eval = np.hstack([X_eval_num, eval_dummies.values])
    else:
        X_train = X_train_num
        X_eval = X_eval_num

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_eval_scaled = scaler.transform(X_eval)

    return X_train_scaled, X_eval_scaled

## **Models**

### **Baseline Model (Random)**

In [50]:
def evaluate_random_baseline(df_train, df_eval, feature_cols, k=10, n_samples=1000, random_state=42):
    rng = np.random.default_rng(random_state)

    X_train_scaled, X_eval_scaled = prepare_scaled_matrices(df_train, df_eval, feature_cols)

    n_train = X_train_scaled.shape[0]
    n_eval = X_eval_scaled.shape[0]
    n_eval_samples = min(n_samples, n_eval)
    eval_indices = rng.choice(n_eval, size=n_eval_samples, replace=False)

    random_sims = []

    for idx in eval_indices:
        query_vec = X_eval_scaled[idx].reshape(1, -1)

        candidates = np.arange(n_train)
        rand_neighbors = rng.choice(candidates, size=k, replace=False)

        rand_vecs = X_train_scaled[rand_neighbors]
        cos_mat = cosine_similarity(query_vec, rand_vecs)[0]
        random_sims.extend(cos_mat.tolist())

    avg_random = float(np.mean(random_sims))
    return avg_random

### **KNN Model**

In [51]:
def evaluate_knn_model(df_train, df_eval, feature_cols, k=10, n_samples=1000, random_state=42):
    rng = np.random.default_rng(random_state)

    X_train_scaled, X_eval_scaled = prepare_scaled_matrices(df_train, df_eval, feature_cols)

    n_eval = X_eval_scaled.shape[0]
    n_eval_samples = min(n_samples, n_eval)
    eval_indices = rng.choice(n_eval, size=n_eval_samples, replace=False)

    knn = NearestNeighbors(metric="euclidean", n_neighbors=k)
    knn.fit(X_train_scaled)

    knn_sims = []

    for idx in eval_indices:
        query_vec = X_eval_scaled[idx].reshape(1, -1)

        distances, indices = knn.kneighbors(query_vec, n_neighbors=k)
        neighbor_indices = indices[0]
        neighbor_vecs = X_train_scaled[neighbor_indices]

        cos_sims = cosine_similarity(query_vec, neighbor_vecs)[0]
        knn_sims.extend(cos_sims.tolist())

    avg_knn = float(np.mean(knn_sims))
    return avg_knn

## **Comparison of Two Models**

### **Tuning Value of K**

In [52]:
ks_to_try = [5, 10, 20]
results_by_k = {}

for k in ks_to_try:
    avg_knn = evaluate_knn_model(train_df, test_df, selected_feature_cols, k=k, n_samples=1000, random_state=42)
    avg_random = evaluate_random_baseline(train_df, test_df, selected_feature_cols, k=k, n_samples=1000, random_state=42)
    improvement = avg_knn - avg_random
    results_by_k[k] = {
        "avg_knn": avg_knn,
        "avg_random": avg_random,
        "improvement": improvement,
    }
    print(f"k = {k}: avg KNN similarity (test) = {avg_knn:.4f}, "
          f"baseline (test) = {avg_random:.4f}, improvement = {improvement:.4f}")

best_k = max(results_by_k, key=lambda kk: results_by_k[kk]["avg_knn"])
print(f"\nBest k based on average cosine similarity (selected features, test set): k = {best_k}")

k = 5: avg KNN similarity (test) = 0.9919, baseline (test) = 0.0034, improvement = 0.9885
k = 10: avg KNN similarity (test) = 0.9899, baseline (test) = 0.0015, improvement = 0.9884
k = 20: avg KNN similarity (test) = 0.9866, baseline (test) = 0.0009, improvement = 0.9857

Best k based on average cosine similarity (selected features, test set): k = 5


### **Test and Training Accuracy**

In [53]:
# Test
final_knn_test = evaluate_knn_model(train_df, test_df, selected_feature_cols, k=best_k, n_samples=2000, random_state=123)
final_random_test = evaluate_random_baseline(train_df, test_df, selected_feature_cols, k=best_k, n_samples=2000, random_state=123)

# Train
final_knn_train = evaluate_knn_model(train_df, train_df, selected_feature_cols, k=best_k, n_samples=2000, random_state=123)
final_random_train = evaluate_random_baseline(train_df, train_df, selected_feature_cols, k=best_k, n_samples=2000, random_state=123)

print("--- Training set ---")
print(f"Average cosine similarity of KNN recommendations (acts as accuracy for similar songs): {final_knn_train:.4f}")
print(f"Average cosine similarity of baseline (random) recommendations: {final_random_train:.4f}")
print(f"Improvement of KNN over random baseline (train): {final_knn_train - final_random_train:.4f}")

print("--- Test set ---")
print(f"Average cosine similarity of KNN recommendations (acts as accuracy for similar songs): {final_knn_test:.4f}")
print(f"Average cosine similarity of baseline (random) recommendations: {final_random_test:.4f}")
print(f"Improvement of KNN over random baseline (test): {final_knn_test - final_random_test:.4f}")

print(f"--- Test vs. Training Accuracy Comparison ---")
print(f"Difference in KNN average similarity (test - train): {final_knn_test - final_knn_train:.4f}")
print(f"Difference in random baseline average similarity (test - train): {final_random_test - final_random_train:.4f}")

--- Training set ---
Average cosine similarity of KNN recommendations (acts as accuracy for similar songs): 0.9940
Average cosine similarity of baseline (random) recommendations: 0.0006
Improvement of KNN over random baseline (train): 0.9934
--- Test set ---
Average cosine similarity of KNN recommendations (acts as accuracy for similar songs): 0.9919
Average cosine similarity of baseline (random) recommendations: -0.0005
Improvement of KNN over random baseline (test): 0.9924
--- Test vs. Training Accuracy Comparison ---
Difference in KNN average similarity (test - train): -0.0021
Difference in random baseline average similarity (test - train): -0.0011


### **Full Feature Set Accuracy vs. Selected Set**

In [54]:
final_full_knn_test = evaluate_knn_model(train_df, test_df, full_feature_cols, k=best_k, n_samples=2000, random_state=123)

print("--- Feature Selection Comparison ---")
print(f"Full feature set ({len(full_feature_cols)} features) - avg KNN cosine similarity (test): {final_full_knn_test:.4f}")
print(f"Selected subset ({len(selected_feature_cols)} features) - avg KNN cosine similarity (test): {final_knn_test:.4f}")

--- Feature Selection Comparison ---
Full feature set (10 features) - avg KNN cosine similarity (test): 0.9869
Selected subset (8 features) - avg KNN cosine similarity (test): 0.9919


## **Flask App**

The flask app can be found at app.py and can be run using 'python app.py' from the appropriate folder.